# Indonesian ensemble rankings creation

This notebook creates a rankings file that ranks supplied tide models at specified coastal locations.

At each location, the NDWI inundation index from **Landsat** satellite imagery is correlated against each tide
model, with the result used to rank the accuracy of each tide model.  A high correlation, i.e. a result that is
close to 1, has a higher ranking than a result that is closer to zero or a negative number.   

### Load packages

In [ ]:
import pandas as pd
import geopandas as gpd
from datacube.utils.dask import start_local_dask
from eo_tides.validation import tide_correlation
from datacube import Datacube

import os
os.environ["USE_PYGEOS"] = "0"

import warnings
warnings.filterwarnings("ignore")

from odc.geo.geom import point
from ensemble_ranking import load_index, corr_to_rankings

In [ ]:
# data access error in landsat causes issue
use_ls = False

In [ ]:
# configuration for landsat access

if use_ls:
    from odc.stac import configure_s3_access

    os.environ["AWS_DEFAULT_REGION"] = "us-west-2"

    if "AWS_NO_SIGN_REQUEST" in os.environ:
        del os.environ["AWS_NO_SIGN_REQUEST"]

    # Configure S3 access including request payer
    _ = configure_s3_access(
        requester_pays=True,
        cloud_defaults=True,
    )

## Dask client
Create local dask client for parallelisation

In [ ]:
# Create local dask client for parallelisation
dask_client = start_local_dask(
    n_workers=8, threads_per_worker=8, mem_safety_margin="2GB",
)

dask_client

### Tide Models

Set model and model data location

In [ ]:
# for running in the sandbox
model_directory = "/home/jovyan/data/coastlines/tide_models/"

In [ ]:
# Check tide models
from eo_tides.utils import list_models

models = list_models(directory=model_directory, show_supported=False)
print(models)

In [ ]:
# Select tide models to be ranked
model_list =[
    'FES2014', 'FES2022', 'EOT20', 'TPXO9-atlas-v5-nc', 'TPXO10-atlas-v2-nc', 'GOT5.6', 'INATIDES'
]

### Points of interest

The points used below are randomly sampled from the `rates_of_change` layer of Indonesia Coastlines v0.0.5, thinned so that points are at least 20km apart. 

In [ ]:
# locations to calculate rankings
poi_file = "ensemble_points_random_sample_20km.geojson"

# output ranking
# file updated after every additional location is assessed
if use_ls: ranking_file = 'ensemble_points_random_sample_20km_rankings_ls.fgb'
else: ranking_file = 'ensemble_points_random_sample_20km_rankings_s2.fgb'
# default to starting from the next point location without ranking
rerun = False

In [ ]:
# Input tide ranking locations
poi = gpd.read_file(poi_file).to_crs('EPSG:4326')
coords = poi.geometry.get_coordinates()
len(coords)

### Tide rankings
For each point/location, correlate NDWI or NMDWI inundation with tide model and rank.

The data is loaded from datacube.

In [ ]:
# function to run for each location

dc = Datacube()

use_s2 = not use_ls
if use_ls: time_range = ("2021", "2025")
else: time_range = ("2023", "2025")

def process_coord(row_dict):
    """
    Define processing for each point location
    """
    
    BUFFER = 2500
    INDEX = "ndwi"

    idx = row_dict["index"]
    x = row_dict["x"]
    y = row_dict["y"]

    print(f"Processing {idx}, {x}, {y}")

    geom = (
        point(x=x, y=y, crs="EPSG:4326")
        .to_crs("utm")
        .buffer(BUFFER)
        .to_crs("EPSG:4326")
    )

    water_index = load_index(
        dc,
        time=time_range,
        geopolygon=geom,
        mask_geopolygon=False,
        max_cloud_cover=40,
        load_ls=use_ls,
        load_s2=use_s2,
        resampling="bilinear",
        index=INDEX,
    )[INDEX]

    corr_df, corr_da = tide_correlation(
        data=water_index,
        model=model_list,
        directory=model_directory,
    )

    return corr_df

### Loop through each location

Since it takes a long term to run all locations, data is saved after each loop.

In [ ]:
# use saved ranking when available
if os.path.exists(ranking_file) and not rerun:
    saved_ranking = gpd.read_file(ranking_file)
    start_idx = saved_ranking['point_idx'].max()+1
else:
    saved_ranking = None
    start_idx = 0

In [ ]:
# load from datacube

# Loop through tide ranking locations and determine tide ranking

rows = [
    {"index": idx, **row.to_dict()}
    for idx, row in coords[start_idx:].iterrows()
]

for row in rows:
    try:
        corr_df = process_coord(row)
        corr_df['point_idx'] = row['index']
        if saved_ranking is None:
            saved_ranking = corr_to_rankings(corr_df)
        else:
            saved_ranking = pd.concat([saved_ranking, corr_to_rankings(corr_df)])
    except:
        print(f"Processing failed: {row['index']}, {row['x']}, {row['y']}")

    saved_ranking.to_file(ranking_file, driver='FlatGeobuf')

### Close Dask client

In [ ]:
dask_client.close()